# 04 — Recommender Prototyping

This notebook prototypes and evaluates the hybrid restaurant recommender.
The recommender blends content-based similarity with Bayesian-adjusted ratings.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import pandas as pd
from dabba.config import get_config
from dabba.data.loaders import load_zomato
from dabba.data.cleaning import clean_zomato
from dabba.features.restaurant_features import add_restaurant_features
from dabba.nlp.sentiment import add_sentiment_scores
from dabba.models.recommender import RestaurantRecommender, bayesian_average

config = get_config()

## 1. Prepare Data

In [ ]:
df = load_zomato(config)
df = clean_zomato(df, config)
df = add_restaurant_features(df, config)
df = add_sentiment_scores(df, config=config)

print(f'Prepared {len(df)} restaurants')

## 2. Bayesian Average Analysis

In [ ]:
if 'rate' in df.columns and 'votes' in df.columns:
    df['bayesian_rating'] = bayesian_average(df['rate'], df['votes'])
    
    # Compare raw vs Bayesian ratings
    comparison = df[['name', 'rate', 'votes', 'bayesian_rating']].head(20)
    comparison = comparison.sort_values('votes', ascending=True)  # Show low-vote restaurants
    
    print('=== Low-Vote Restaurants: Raw vs Bayesian Rating ===')
    print(comparison.head(10).to_string(index=False))
    print('\n→ Bayesian average pulls low-vote restaurants toward the global mean')

## 3. Initialize Recommender

In [ ]:
feature_cols = ['votes_log', 'cost_for_two']
feature_cols += [c for c in df.columns if c.startswith('cuisine_')][:10]  # Top 10 cuisines

rec = RestaurantRecommender(df, feature_cols, config)
print(f'Recommender initialized with {len(rec.df)} restaurants and {len(feature_cols)} features')

## 4. Test Recommendations

In [ ]:
# Test: North Indian, budget ₹500, Koramangala
print('=== Recommendations: North Indian, Budget ₹500, Koramangala ===')
recs = rec.recommend(cuisine='North Indian', budget=500, area='Koramangala', top_n=5)
if not recs.empty:
    print(recs[['name', 'rate', 'cost_for_two', 'combined_score']].to_string(index=False))
else:
    print('No matches — try broader filters')

In [ ]:
# Test: Italian, any budget, any area
print('=== Recommendations: Italian, Any Budget ===')
recs = rec.recommend(cuisine='Italian', top_n=5)
if not recs.empty:
    print(recs[['name', 'rate', 'cost_for_two', 'location', 'combined_score']].to_string(index=False))
else:
    print('No matches')

In [ ]:
# Test: High budget, MG Road
print('=== Recommendations: Premium, MG Road ===')
recs = rec.recommend(budget=2000, area='MG Road', top_n=5)
if not recs.empty:
    print(recs[['name', 'rate', 'cost_for_two', 'cuisines', 'combined_score']].to_string(index=False))
else:
    print('No matches')

## Key Takeaways

- The hybrid approach combines content similarity with popularity signals
- Bayesian average prevents low-vote restaurants from dominating
- The recommender can be tuned by adjusting feature weights